# Задача 3, этап 3: одинаковый бюджет шагов и несколько сидов

Параметр исследования (alpha у DSAC, ent_coef у PPO) на картах 10x10 — лучший по перебору `task2_tuning.ipynb`.
Бюджет шагов среды N для каждой среды — больший из «шагов до плато» у PPO и DSAC с памятью (с выбранными значениями).
Все прогоны учатся ровно N шагов, поэтому оба алгоритма доучены, а бюджет у них одинаковый.
Каждая конфигурация — 3 сида (0, 1, 2); на графиках среднее по сидам и разброс между ними. Результаты — `outputs/task3_v3/`.

In [1]:
import csv
import json
import os
import sys
from pathlib import Path

# работаем из корня репозитория: там лежат src/ и outputs/
if Path.cwd().name == "tasks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src import utils
from src.envs import load_mnist, make_env
from src.models import DSAC, PPO

In [2]:
# режим запуска: при обычном открытии ноутбука — полное обучение всех экспериментов и все графики
# на сервере скрипт задаёт переменные, чтобы запускать эксперименты параллельно
RUN_MODE = os.environ.get("RUN_MODE", "full")  # full или smoke (короткая проверка)
RUN_ONLY = os.environ.get("RUN_ONLY", "")  # имя одного эксперимента, пусто — все
RUN_STAGES = os.environ.get("RUN_STAGES", "train,plots").split(",")

torch.set_num_threads(1)  # прогоны идут параллельно, каждому хватит одного потока
CUDA = torch.cuda.is_available()
print("режим:", RUN_MODE, "| эксперимент:", RUN_ONLY or "все", "| этапы:", RUN_STAGES, "| CUDA:", CUDA)

режим: full | эксперимент: ppo_mnist_env4_gru_seed1 | этапы: ['train'] | CUDA: True


In [3]:
SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — полностью наблюдаемая
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}
del SETUPS[1]  # 25 цветов не помещаются в 10 цифр

## Гиперпараметры

In [4]:
ppo_cfg = {
    "device": "cpu",  # PPO на видеокарте не быстрее (замер: 479 против 429 шагов в секунду)
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сеть и старты эпизодов
    "memory": True,  # True — GRU, False — MLP без памяти
    "episodes_per_update": 10,  # сколько эпизодов собрать перед обновлением
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "save_every": 50,  # как часто сохранять метрики и веса на диск
}

In [5]:
dsac_cfg = {
    "device": "cuda" if CUDA else "cpu",  # DSAC на видеокарте в 3 раза быстрее (167 против 52 шагов в секунду)
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "soft_q": True,  # мягкая Q (энтропия в цели критика) или обычная
    "buffer_episodes": 5000,  # сколько последних эпизодов помнить
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,  # скорость, с которой target-критики догоняют обычных
    "lr": 3e-4,
    "alpha": 0.005,  # температура: вес энтропии
    "hidden": 128,
    "save_every": 25,
}

In [6]:
ALPHAS = [0.01, 0.02, 0.05]  # alpha DSAC в переборе (0.005 — значение из этапа 2)
ENT_COEFS = [0.001, 0.03, 0.1]  # ent_coef PPO в переборе (0.01 — значение из этапа 2)

EVAL_EPISODES = {"full": 100, "smoke": 5}

OUT = "outputs/task3_v3" if RUN_MODE == "full" else "runs/smoke/task3_v3"

## Параметр исследования и бюджет шагов

In [7]:
def pick_best(candidates):
    # candidates: {значение параметра: папка прогона}; лучшее — с наибольшей средней отдачей в жадной оценке
    best_value, best_path, best_return = None, None, -1.0
    for value, path in candidates.items():
        result = json.loads((Path(path) / "eval.json").read_text())
        if result["greedy"]["mean_return"] > best_return:
            best_value, best_path, best_return = value, path, result["greedy"]["mean_return"]
    return best_value, best_path


def steps_to_plateau(path):
    # сколько шагов среды прогону понадобилось до плато (если не сошёлся — все его шаги)
    status = json.loads((Path(path) / "status.json").read_text())
    if status["converged"]:
        return status["converged_at"]["env_steps"]
    return status["env_steps"]

In [8]:
V2 = "outputs/task3_v2" if RUN_MODE == "full" else "runs/smoke/task3_v2"
V2_TASK2 = "outputs/task2_v2" if RUN_MODE == "full" else "runs/smoke/task2_v2"
TUNING = "outputs/task2_tuning" if RUN_MODE == "full" else "runs/smoke/task2_tuning"

# на 5x5 значения как в этапе 2; на 10x10 — лучшие по перебору (на обычных картах, без MNIST)
BEST = {"ppo": {}, "dsac": {}}
TUNED = {"ppo": {}, "dsac": {}}  # прогон с выбранным значением
for i in SETUPS:
    BEST["ppo"][i] = {"ent_coef": 0.01}
    BEST["dsac"][i] = {"alpha": 0.005}
    TUNED["ppo"][i] = f"{V2_TASK2}/runs/ppo_env{i}_gru_seed0"
    TUNED["dsac"][i] = f"{V2_TASK2}/runs/dsac_env{i}_gru_seed0"
for i in [3, 4]:
    candidates = {0.01: TUNED["ppo"][i]}
    for value in ENT_COEFS:
        candidates[value] = f"{TUNING}/runs/ppo_env{i}_gru_ent{value}"
    value, TUNED["ppo"][i] = pick_best(candidates)
    BEST["ppo"][i] = {"ent_coef": value}
    candidates = {0.005: TUNED["dsac"][i]}
    for value in ALPHAS:
        candidates[value] = f"{TUNING}/runs/dsac_env{i}_gru_alpha{value}"
    value, TUNED["dsac"][i] = pick_best(candidates)
    BEST["dsac"][i] = {"alpha": value}
print("выбранные значения:", BEST)

# бюджет N — больший из «шагов до плато» у PPO и DSAC с памятью на MNIST-картах из этапа 2
BUDGET = {}
for i in SETUPS:
    BUDGET[i] = max(steps_to_plateau(f"{V2}/runs/ppo_mnist_env{i}_gru_seed0"),
                    steps_to_plateau(f"{V2}/runs/dsac_mnist_env{i}_gru_seed0"))
print("бюджет шагов по средам:", BUDGET)

SEEDS = [0, 1, 2]

выбранные значения: {'ppo': {2: {'ent_coef': 0.01}, 3: {'ent_coef': 0.001}, 4: {'ent_coef': 0.001}}, 'dsac': {2: {'alpha': 0.005}, 3: {'alpha': 0.01}, 4: {'alpha': 0.01}}}
бюджет шагов по средам: {2: 15630, 3: 171264, 4: 738511}


## Список экспериментов

In [9]:
EXPERIMENTS = []
for seed in SEEDS:
    for i in SETUPS:
        for memory in [True]:
            tag = "gru" if memory else "mlp"
            for algo, base in [("ppo", ppo_cfg), ("dsac", dsac_cfg)]:
                name = f"{algo}_mnist_env{i}_{tag}_seed{seed}"
                cfg = {**base, **SETUPS[i], "mnist": True, "memory": memory, "seed": seed, "episodes": 1_000_000, "max_env_steps": BUDGET[i], **BEST[algo][i],
                       "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": name}
                EXPERIMENTS.append({"name": name, "algo": algo, "cfg": cfg})

for exp in EXPERIMENTS:
    print(exp["name"], exp["cfg"]["device"])

ppo_mnist_env2_gru_seed0 cpu
dsac_mnist_env2_gru_seed0 cuda
ppo_mnist_env3_gru_seed0 cpu
dsac_mnist_env3_gru_seed0 cuda
ppo_mnist_env4_gru_seed0 cpu
dsac_mnist_env4_gru_seed0 cuda
ppo_mnist_env2_gru_seed1 cpu
dsac_mnist_env2_gru_seed1 cuda
ppo_mnist_env3_gru_seed1 cpu
dsac_mnist_env3_gru_seed1 cuda
ppo_mnist_env4_gru_seed1 cpu
dsac_mnist_env4_gru_seed1 cuda
ppo_mnist_env2_gru_seed2 cpu
dsac_mnist_env2_gru_seed2 cuda
ppo_mnist_env3_gru_seed2 cpu
dsac_mnist_env3_gru_seed2 cuda
ppo_mnist_env4_gru_seed2 cpu
dsac_mnist_env4_gru_seed2 cuda


## Обучение и оценка

In [10]:
def run_experiment(exp):
    # прогон, который уже закончился, второй раз не учим
    status_file = Path(OUT) / "runs" / exp["name"] / "status.json"
    if status_file.exists() and json.loads(status_file.read_text())["status"] == "completed":
        print("уже готово:", exp["name"])
        return

    if exp["algo"] == "ppo":
        alg = PPO(exp["cfg"])
    else:
        alg = DSAC(exp["cfg"])
    alg.train()

    # оценка на одних и тех же стартах (сиды 10000, 10001, ...): жадно и с сэмплированием действий
    n = EVAL_EPISODES[RUN_MODE]
    greedy = alg.evaluate(n, greedy=True)
    sampled = alg.evaluate(n, greedy=False)
    result = {"budget": RUN_MODE,
              "greedy": utils.eval_summary(greedy, alg.cfg["gamma"], alg.optimal),
              "stochastic": utils.eval_summary(sampled, alg.cfg["gamma"], alg.optimal)}
    utils.save_eval(alg.run_dir, result)

    s = result["stochastic"]
    title = f"{exp['name']}: успех {s['success_rate']:.2f}, длина {s['mean_length']:.1f} (оптимум {s['optimal_length']:.1f})"
    utils.save_eval_pictures(alg.env, sampled, Path(OUT) / "eval", exp["name"], title)

In [11]:
if "train" in RUN_STAGES:
    for exp in EXPERIMENTS:
        if RUN_ONLY == "" or RUN_ONLY == exp["name"]:
            run_experiment(exp)

episode 100 | steps 14890 | return 0.383 (opt 0.938) | length 148.9 | entropy 1.33


episode 200 | steps 24836 | return 0.505 (opt 0.938) | length 99.5 | entropy 1.29


episode 300 | steps 35068 | return 0.539 (opt 0.938) | length 102.3 | entropy 1.24


episode 400 | steps 41826 | return 0.631 (opt 0.938) | length 67.6 | entropy 1.18


episode 500 | steps 46757 | return 0.703 (opt 0.938) | length 49.3 | entropy 1.06


...

episode 57600 | steps 736292 | return 0.914 (opt 0.938) | length 10.1 | entropy 0.08


episode 57700 | steps 737341 | return 0.910 (opt 0.938) | length 10.5 | entropy 0.09


episode 57800 | steps 738371 | return 0.912 (opt 0.938) | length 10.3 | entropy 0.09


бюджет шагов 738511 исчерпан: 738573 шагов


saved outputs/task3_v3/eval/ppo_mnist_env4_gru_seed1.gif


## Графики: среднее по сидам

In [12]:
plt.style.use("seaborn-v0_8-darkgrid")


def load_metrics(run_path):
    # metrics.csv прогона: словарь {колонка: массив чисел}; None, если прогона нет
    path = Path(run_path) / "metrics.csv"
    if not path.exists():
        print("нет данных:", run_path)
        return None
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    data = {}
    for key in rows[0]:
        values = [{"True": 1.0, "False": 0.0, "": np.nan}.get(row[key], row[key]) for row in rows]
        data[key] = np.array(values, dtype=float)
    return data

In [13]:
def rolling_mean(values, window=50):
    # среднее за последние window эпизодов (пропуски nan не учитываем)
    result = []
    for i in range(len(values)):
        last = values[max(0, i - window + 1):i + 1]
        last = last[~np.isnan(last)]
        result.append(np.mean(last) if len(last) > 0 else np.nan)
    return np.array(result)

In [14]:
def seed_curve(path, key, grid):
    # сглаженная кривая одного прогона, пересчитанная на общую сетку шагов среды
    data = load_metrics(path)
    if data is None:
        return None
    return np.interp(grid, data["env_steps"], rolling_mean(data[key]), right=np.nan)


def plot_seeds(runs, key, ylabel, filename):
    # runs: {"Env2": {"PPO": [пути по сидам], "DSAC": [...]}, ...}; линия — среднее по сидам,
    # полоса — стандартное отклонение между сидами (это настоящий разброс между запусками)
    colors = ["tab:red", "tab:blue", "tab:green"]
    fig, axes = plt.subplots(1, len(runs), figsize=(5.5 * len(runs), 4.3), squeeze=False)
    for ax, env_name in zip(axes[0], runs):
        grid = np.linspace(0, BUDGET[int(env_name[-1])], 200)
        for (label, paths), color in zip(runs[env_name].items(), colors):
            curves = [seed_curve(path, key, grid) for path in paths]
            curves = [c for c in curves if c is not None]
            if len(curves) == 0:
                continue
            mean = np.nanmean(curves, axis=0)
            std = np.nanstd(curves, axis=0)
            ax.plot(grid, mean, color=color, linewidth=2, label=f"{label} ({len(curves)} сида)")
            ax.fill_between(grid, mean - std, mean + std, color=color, alpha=0.25)
        ax.set_title(env_name)
        ax.set_xlabel("шаги среды")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    Path(OUT, "figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(Path(OUT) / "figures" / filename, dpi=150, bbox_inches="tight")
    plt.show()

In [15]:
if "plots" in RUN_STAGES:
    runs = {}
    for i in SETUPS:
        runs[f"Env{i}"] = {
            "PPO + GRU": [f"{OUT}/runs/ppo_mnist_env{i}_gru_seed{s}" for s in SEEDS],
            "DSAC + GRU": [f"{OUT}/runs/dsac_mnist_env{i}_gru_seed{s}" for s in SEEDS],
        }
    for key, ylabel in [("return", "средняя дисконтированная отдача"), ("success", "доля успешных эпизодов"),
                        ("length", "длина эпизода"), ("entropy", "энтропия стратегии"),
                        ("explore_prob", "вероятность нежадного действия"), ("loss", "полный лосс")]:
        plot_seeds(runs, key, ylabel, f"ppo_vs_dsac_{key}.png")

## Итоговая оценка по сидам

In [16]:
def convergence_rows(run_paths):
    # когда каждый прогон вышел на плато (или упёрся в потолок эпизодов) и как он потом сыграл в оценке
    rows = []
    for path in run_paths:
        status_file = Path(path) / "status.json"
        if not status_file.exists():
            rows.append({"run": Path(path).name, "converged": "pending"})
            continue
        status = json.loads(status_file.read_text(encoding="utf-8"))
        row = {"run": Path(path).name, "converged": status.get("converged"),
               "episodes": status["episodes_done"], "env_steps": status.get("env_steps"),
               "time_min": round(status.get("time_sec", 0) / 60, 1)}
        eval_file = Path(path) / "eval.json"
        if eval_file.exists():
            result = json.loads(eval_file.read_text(encoding="utf-8"))
            row["greedy_success"] = round(result["greedy"]["success_rate"], 3)
            row["greedy_length"] = round(result["greedy"]["mean_length"], 1)
            row["optimal_length"] = round(result["greedy"]["optimal_length"], 2)
        rows.append(row)
    return rows


def save_rows(rows, filename):
    Path(OUT, "tables").mkdir(parents=True, exist_ok=True)
    fields = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with open(Path(OUT) / "tables" / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)

In [17]:
def seeds_rows(groups):
    # groups: {"ppo_env2_gru": [пути по сидам], ...}; среднее и разброс итоговой оценки по сидам
    rows = []
    for name, paths in groups.items():
        results = []
        for path in paths:
            eval_file = Path(path) / "eval.json"
            if eval_file.exists():
                results.append(json.loads(eval_file.read_text(encoding="utf-8")))
        if len(results) == 0:
            rows.append({"config": name, "seeds": 0})
            continue
        row = {"config": name, "seeds": len(results)}
        for mode in ["greedy", "stochastic"]:
            success = [r[mode]["success_rate"] for r in results]
            length = [r[mode]["mean_length"] for r in results]
            row[f"{mode}_success_mean"] = round(np.mean(success), 3)
            row[f"{mode}_success_std"] = round(np.std(success), 3)
            row[f"{mode}_length_mean"] = round(np.mean(length), 1)
            row[f"{mode}_length_std"] = round(np.std(length), 1)
        row["optimal_length"] = round(results[0]["greedy"]["optimal_length"], 2)
        rows.append(row)
    return rows

In [18]:
if "plots" in RUN_STAGES:
    groups = {}
    for exp in EXPERIMENTS:
        config = exp["name"].rsplit("_seed", 1)[0]
        groups.setdefault(config, []).append(f"{OUT}/runs/{exp['name']}")
    save_rows(seeds_rows(groups), "eval_seeds.csv")